# Pronóstico de temperatura con LSTM *one-shot*

Este tutorial reproduce el flujo de `s11_4_ts_prediction_lstm_jena_one_shot.py`: usa las temperaturas diarias de Jena para entrenar una red LSTM con datos de 2009–2016 y predecir **todo 2017 en una sola salida** (*one-shot*). Al final compara el pronóstico con las observaciones mediante MAE, RMSE y R², y representa los resultados.

> **Datos y atribución:** el conjunto de datos climáticos de Jena es proporcionado por el Max Planck Institute for Biogeochemistry bajo la licencia [Creative Commons Attribution 4.0 (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/). Coloca `jena_daily_temp_2009_2017.csv` en `time_series/data/` antes de ejecutar el notebook.

Para ejecutar un Jupyter Notebook (.ipynb) en Visual Studio Code necesitas:
- las extensiones de Jupyter y Python en VSCode
- el paquete ipykernel (`pip install ipykernel`)

## 1. Preparación e importaciones

Configuramos los mensajes de TensorFlow antes de importar Keras. Las bibliotecas cubren lectura de datos, escalado, métricas, gráficos y persistencia del modelo y del escalador.

In [1]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import Input
from keras.layers import LSTM, Dense, Dropout
from keras.models import Sequential, load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

## 2. Configuración del experimento

`WINDOW` indica cuántos días de historia recibe el modelo y `HORIZON` cuántos días predice de una vez. El modelo recibe una ventana de 365 días y genera los 365 valores de 2017 en una única pasada. `STRIDE` controla cuánto se desplaza la ventana al construir los ejemplos de entrenamiento.

Las rutas se resuelven tanto si se ejecuta Jupyter desde la raíz del repositorio como desde `time_series/`.

In [ ]:
WINDOW = 365
HORIZON = 365
STRIDE = 1

LSTM_UNITS = 50
DROPOUT = 0.2
EPOCHS = 20
BATCH_SIZE = 32

cwd = Path.cwd()
if cwd.name == "time_series":
    time_series_dir = cwd
elif (cwd / "time_series").is_dir():
    time_series_dir = cwd / "time_series"
else:
    time_series_dir = cwd

data_file = time_series_dir / "data" / "jena_daily_temp_2009_2017.csv"
model_dir = time_series_dir / "models"
output_dir = time_series_dir / "output"
model_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

model_name = (
    f"lstm_u{LSTM_UNITS}_e{EPOCHS}_b{BATCH_SIZE}_w{WINDOW}_h{HORIZON}"
    f"_s{STRIDE}_d{round(100 * DROPOUT)}"
)
model_file = model_dir / f"{model_name}.keras"
scaler_file = model_dir / f"{model_name}.scaler"

print(f"Datos: {data_file}")
print(f"Modelo: {model_file}")

## 3. Carga de datos y separación temporal

Usamos la columna `T (degC)` como serie objetivo y `Date Time` como índice temporal. La separación es cronológica: 2009–2016 para entrenamiento y 2017 para prueba. Esta separación evita entrenar con observaciones futuras respecto al período evaluado.

In [ ]:
if not data_file.is_file():
    raise FileNotFoundError(
        f"No se encontró el conjunto de datos: {data_file}. "
        "Descarga el CSV diario de temperaturas de Jena y colócalo en "
        "time_series/data/jena_daily_temp_2009_2017.csv."
    )

df = pd.read_csv(data_file)
df.head(3)

In [ ]:
df["Date Time"] = pd.to_datetime(
    df["Date Time"], format="%d.%m.%Y %H:%M:%S"
)
ts = df.set_index("Date Time")["T (degC)"]

train_ts = ts["2009":"2016"]
test_ts = ts["2017"]

print(f"Observaciones de entrenamiento: {len(train_ts)}")
print(f"Observaciones de prueba (2017): {len(test_ts)}")
print(f"Período de entrenamiento: {train_ts.index.min()} — {train_ts.index.max()}")
print(f"Período de prueba: {test_ts.index.min()} — {test_ts.index.max()}")

## 4. Preparación de ventanas

Cada ejemplo de entrada contiene `WINDOW` temperaturas consecutivas y su objetivo contiene las `HORIZON` temperaturas siguientes. El escalador se **ajusta solo con el conjunto de entrenamiento** y después se reutiliza para transformar los datos; así evitamos que información de 2017 influya en el preprocesamiento.

In [ ]:
def create_sequences(data, window, horizon, stride=1):
    """Construye pares de ventanas de entrada y horizontes objetivo."""
    X, y = [], []
    for i in range(0, len(data) - window - horizon + 1, stride):
        X.append(data[i : i + window])
        y.append(data[i + window : i + window + horizon])
    return np.array(X), np.array(y)


scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_ts.to_numpy().reshape(-1, 1))
print("Primeras temperaturas escaladas:")
print(train_scaled[:3])

In [ ]:
X_train, y_train = create_sequences(
    train_scaled, WINDOW, HORIZON, stride=STRIDE
)

# La salida Dense(HORIZON) espera (muestras, horizonte), sin la dimensión de característica.
y_train = y_train.squeeze(-1)
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

## 5. Construcción, entrenamiento y reutilización del modelo

La arquitectura tiene una capa LSTM, una capa Dropout y una capa densa de 365 unidades. La capa final produce el horizonte entero directamente: no realimenta predicciones intermedias. Si existen **tanto** el archivo del modelo como el del escalador con la configuración actual, se cargan para evitar entrenar de nuevo; en caso contrario, se entrena y se guardan ambos.

In [ ]:
if model_file.is_file() and scaler_file.is_file():
    print("Cargando modelo y escalador existentes")
    model = load_model(model_file)
    scaler = joblib.load(scaler_file)
    train_scaled = scaler.transform(train_ts.to_numpy().reshape(-1, 1))
    model.summary()
else:
    model = Sequential(
        [
            Input(shape=(WINDOW, 1)),
            LSTM(LSTM_UNITS),
            Dropout(DROPOUT),
            Dense(HORIZON),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae", "r2_score"],
    )
    model.summary()

In [ ]:
if not (model_file.is_file() and scaler_file.is_file()):
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
    )

    final_rmse = np.sqrt(history.history["loss"][-1])
    final_mae = history.history["mae"][-1]
    final_r2 = history.history["r2_score"][-1]
    val_rmse = np.sqrt(history.history["val_loss"][-1])
    val_mae = history.history["val_mae"][-1]
    val_r2 = history.history["val_r2_score"][-1]

    print("Métricas de entrenamiento (valores normalizados):")
    print(f"  MAE  = {final_mae:.3f}")
    print(f"  RMSE = {final_rmse:.3f}")
    print(f"  R²   = {final_r2:.3f}")
    print("Métricas de validación (valores normalizados):")
    print(f"  MAE  = {val_mae:.3f}")
    print(f"  RMSE = {val_rmse:.3f}")
    print(f"  R²   = {val_r2:.3f}")

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="Entrenamiento")
    plt.plot(history.history["val_loss"], label="Validación")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (MSE)")
    plt.title("Pérdida de entrenamiento y validación")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"{model_name}_loss_curve.png", dpi=300)
    plt.show()

    model.save(model_file)
    joblib.dump(scaler, scaler_file)
    print(f"Modelo guardado en: {model_file}")
    print(f"Escalador guardado en: {scaler_file}")

### Interpretación de la curva de pérdida

Una pérdida de entrenamiento y validación que desciende sugiere que el modelo está aprendiendo. Si la de entrenamiento baja mientras la de validación sube, puede haber sobreajuste; si ambas se estancan pronto, puede haber subajuste o convergencia. Las métricas de entrenamiento/validación anteriores están en escala normalizada, por lo que no se deben comparar directamente con las métricas finales en grados Celsius.

## 6. Pronóstico y evaluación en 2017

La entrada para la predicción son los últimos 365 días del período de entrenamiento. El modelo produce los 365 valores del horizonte en una sola llamada. Invertimos el escalado antes de calcular las métricas, de modo que MAE y RMSE se expresan en °C.

In [ ]:
window_data = train_scaled[-WINDOW:].copy()
X_input = window_data.reshape(1, WINDOW, 1)

y_pred_scaled = model.predict(X_input, verbose=0)
print(f"Forma de la predicción escalada: {y_pred_scaled.shape}")

y_pred = scaler.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1))
y_true = test_ts.to_numpy()

print(f"Forma de y_pred: {y_pred.shape}")
print(f"Forma de y_true: {y_true.shape}")

In [ ]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("Métricas de prueba (2017):")
print(f"  MAE  = {mae:.3f} °C")
print(f"  RMSE = {rmse:.3f} °C")
print(f"  R²   = {r2:.3f}")

prediction_ts = pd.Series(y_pred.ravel(), index=test_ts.index)
observed_ts = pd.Series(y_true.ravel(), index=test_ts.index)

## 7. Visualización del pronóstico

El primer gráfico muestra el historial de entrenamiento y el pronóstico de 2017; el segundo amplía el período de prueba para comparar las temperaturas observadas y predichas día a día. Los gráficos se guardan en `time_series/output/`.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train_ts.index, train_ts.values, linewidth=0.5, alpha=0.5, label="Datos de entrenamiento")
plt.plot(test_ts.index, test_ts.values, linewidth=0.5, label="Observado 2017")
plt.plot(prediction_ts.index, prediction_ts.values, linewidth=1, label="Pronóstico LSTM")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.title("Temperatura diaria histórica y pronóstico LSTM (one-shot)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / f"{model_name}_forecast.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_ts.index, test_ts.values, linewidth=1, label="Observado 2017")
plt.plot(prediction_ts.index, prediction_ts.values, linewidth=1, label="Pronóstico LSTM")
plt.title("Pronóstico diario de temperatura (2017)")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(output_dir / f"{model_name}_forecast_zoom.png", dpi=300)
plt.show()

## Resumen

El experimento escala la serie con parámetros aprendidos únicamente de 2009–2016, genera ejemplos deslizantes, entrena (o reutiliza) una LSTM que predice el año de prueba de una sola vez y evalúa en la escala original. Para volver a entrenar con una configuración ya guardada, elimina manualmente el modelo y su archivo `.scaler` correspondientes en `time_series/models/`.